# Core segmentation and feature workflow

This compact example uses synthetic pixels to demonstrate the pixel-based analysis API.

In [ ]:
import numpy as np
from nuclear_imaging_core.segmentation import segment3d_multiotsu_simple
from nuclear_imaging_core.features.two_d.shape_hull_features import hullFeatures2D
from nuclear_imaging_core.measurements import normalized_radial_profile, region_feature_table
from nuclear_imaging_core.graph.dense_regions import DenseRegionConfig, segment_dense_regions_frame
from nuclear_imaging_core.graph.graph_build import build_frame_graph_bundle


In [ ]:
volume = np.zeros((5, 64, 64), dtype=np.float32)
volume[:, 12:52, 12:52] = 0.5
volume[:, 24:40, 24:40] = 1.0
labels, mask = segment3d_multiotsu_simple(volume, min_size=16)
labels.shape, labels.dtype, np.unique(labels)


In [ ]:
features_2d = hullFeatures2D(mask[2])
features_2d


In [ ]:
object_table = region_feature_table(volume, labels)
radial_table = normalized_radial_profile(volume, mask, bins=5)
object_table, radial_table


In [ ]:
graph_input = volume[2]
graph_config = DenseRegionConfig(
    threshold_mode='quantile', threshold_quantile=80,
    with_peaks=True, with_boundary_nodes=True, min_region_size_px=9,
)
dense_regions = segment_dense_regions_frame(graph_input, 'reference', graph_config)
graph_bundle = build_frame_graph_bundle(dense_regions, 'reference')
graph_bundle.graph_attrs
